In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import jax
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

import altair as alt

from mixres.sim import PoissonEnveloped1D, make_aggregation_matrix
from mixres.models._inference import run_inference_mcmc

from bspline1d_utils import (
    make_bspline_basis,
    make_gmrf_precision,
    gmrf_covariance_cholesky,
)
from interval_utils import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_interval_sum_matrix,
)

## Data Generation

In [21]:
# Generate the dataset
# left_limits = [0, 0, 5, 8, 10, 15, 18, 20, 25, 30, 35, 40, 50, 60, 70, 80, 90]
# right_limits = [5, 12, 15, 25, 20, 30, 35, 40, 45, 50, 55, 60, 70, 80, 90, 95, 99]

left_limits = [0, 10, 20, 30, 50, 70, 80, 90]
right_limits = [9, 19, 29, 49, 69, 79, 89, 99]

intervals = [
    f"[{left},{right}{']' if right in {5, 15, 25, 30, 40, 50, 60, 70, 80, 90, 99} else ')'}"
    for left, right in zip(left_limits, right_limits)
]
print("Intervals:", intervals)

df_obs, df_rates = PoissonEnveloped1D(
    obs_intervals=intervals, n_observations=20, seed=42
).generate(return_true_rates=True)

# Expanded interval dataframe for plotting
df_expanded = expand_interval_dataframe(df_obs, interval_col="obs_interval", x_col="x")

Intervals: ['[0,9)', '[10,19)', '[20,29)', '[30,49)', '[50,69)', '[70,79)', '[80,89)', '[90,99]']


In [22]:
def summarize_interval_segments(df_segment: pd.DataFrame) -> pd.DataFrame:
    return (
        df_segment[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )


def plot_data(df_true: pd.DataFrame, df_segment: pd.DataFrame) -> alt.Chart:
    source_segment = summarize_interval_segments(df_segment)

    chart_true = (
        alt.Chart(df_true)
        .mark_line(color="red")
        .encode(x="x", y=alt.Y("lambda:Q", title="lambda"))
    )

    chart_segment = (
        alt.Chart(source_segment)
        .mark_rule(color="orange", strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="lambda"),
        )
    )

    return chart_true + chart_segment

In [23]:
plot_data(df_rates, df_expanded)

alt.LayerChart(...)

In [24]:
W = make_aggregation_matrix(left_limits, right_limits)
Phi = make_bspline_basis(x=np.linspace(0, 100, 100), n_basis=40, boundary_ext=0)
Q = make_gmrf_precision(Phi.shape[1], order=2, tau=1.0, jitter=1e-4)
K = gmrf_covariance_cholesky(Q)

In [25]:
def make_fisher_information(
    lambda_: np.ndarray, W: np.ndarray, Phi: np.ndarray, K: np.ndarray | None = None
) -> np.ndarray:
    """
    Computes the Fisher information matrix for the Poisson model given the rate,
    aggregation matrix, and B-spline basis.

    If `K` is given (the Cholesky factor of a coefficient covariance prior,
    Sigma = K @ K.T), the Fisher information is reparametrized into the
    whitened coordinates beta = K @ alpha, alpha ~ N(0, I): I_alpha = K.T @
    I_beta @ K. This lets the identifiability analysis (eigendecomposition
    into image/null space) reflect a spatial smoothness prior instead of
    treating coefficient directions as independent.
    """

    inv_eta = 1 / (W @ lambda_ / W.sum(axis=1))
    Dinv_eta = np.diag(inv_eta)
    Dlam = np.diag(lambda_)

    I = Phi.T @ Dlam @ W.T @ Dinv_eta @ W @ Dlam @ Phi
    if K is not None:
        I = K.T @ I @ K
    return I

## Subspace Decomposition

### Setting GMRF-Regularized Fisher Information

Settings 1-3 all treat coefficient directions as a priori independent: the eigendecomposition of `Phi.T @ Dlam @ W.T @ Dinv_eta @ W @ Dlam @ Phi` splits `beta`-space into image/null subspaces with no notion that neighboring spline coefficients should be spatially correlated. This setting instead places a Gaussian Markov Random Field (GMRF) smoothness prior on `beta`, expressed via its covariance's Cholesky factor `K` (`Sigma = K @ K.T`, from `make_gmrf_precision` / `gmrf_covariance_cholesky` in `bspline1d_utils`), and folds it into the Fisher information as `K.T @ I @ K` -- the Fisher information of the whitened coordinates `alpha` under the reparametrization `beta = K @ alpha`, `alpha ~ N(0, I)`.

We reuse the empirical point estimate `lambda_hat` from Setting 2, so this setting differs from Setting 2 only in reparametrizing coefficient space by the GMRF prior before eigendecomposing -- not in how `lambda_` (and hence `Dlam`) is built. `PrIm_reg`/`PrKer_reg` therefore live in `alpha`-space, not raw `beta`-space: reconstructing `f` requires `Phi @ K @ (...)` rather than `Phi @ (...)`.

In [26]:
df_eta_hat = df_obs.groupby("obs_interval_id")["y"].agg(N="size", y="sum").reset_index()
eta_hat = (df_eta_hat["y"] / df_eta_hat["N"]).to_numpy()

interval_avg_op = W / W.sum(axis=1, keepdims=True)
lambda_hat = np.linalg.pinv(interval_avg_op) @ eta_hat
assert np.allclose(interval_avg_op @ lambda_hat, eta_hat)

In [27]:
I = make_fisher_information(lambda_hat, W, Phi, K=K)

eigvals, eigvecs = LA.eigh(I)

tol = eigvals.max() * len(eigvals) * np.finfo(eigvals.dtype).eps
PrIm = eigvecs[:, eigvals > tol]
PrKer = eigvecs[:, eigvals <= tol]

print(PrIm.shape, PrKer.shape)

(40, 8) (40, 32)


In [28]:
print("tolerance threshold:", tol)
print("eigvals:", eigvals)

tolerance threshold: 5.29320717670347e-09
eigvals: [-1.29169858e-10 -8.64755710e-11 -5.88894237e-11 -5.73152231e-11
 -4.85738462e-11 -4.56896562e-11 -3.72030967e-11 -2.48478619e-11
 -1.83544923e-11 -1.18662883e-11 -8.73795705e-12 -2.26842635e-12
 -1.86823038e-12 -1.38631110e-12 -4.44630542e-13 -2.56305739e-13
 -1.20072339e-13  8.40247323e-16  4.29787962e-15  6.33800575e-13
  4.78136133e-12  4.98860045e-12  1.09521472e-11  1.31927434e-11
  2.48104814e-11  3.03996404e-11  3.63679000e-11  4.37195386e-11
  5.24996486e-11  5.69542876e-11  7.26644028e-11  1.25691184e-10
  6.67185441e+01  2.60243157e+02  5.03893725e+02  1.97237387e+03
  4.42352324e+03  5.88068869e+04  1.58851086e+05  5.95962147e+05]


In [29]:
alpha_Im = PrIm @ np.random.normal(0, 1, size=(PrIm.shape[1], 1))
alpha_Ker = PrKer @ np.random.normal(0, 1, size=(PrKer.shape[1], 1))
f_Im = Phi @ K @ alpha_Im
f_Ker = Phi @ K @ alpha_Ker

source = pd.DataFrame(
    {
        "x": np.linspace(0, 100, 100),
        "f_Im": f_Im.flatten(),
        "f_Ker": f_Ker.flatten(),
    }
)
chart = alt.Chart(source).mark_line(color="green").encode(x="x", y="f_Im")
chart_Ker = alt.Chart(source).mark_line(color="blue").encode(x="x", y="f_Ker")
chart + chart_Ker

alt.LayerChart(...)

## Result Plotting Helpers

Shared helpers for the 4 downstream experiments' result plots, so each experiment's own plot and the final side-by-side comparison are built from the same code.

In [30]:
def summarize_posterior_lam(
    mcmc_samples: dict, x: np.ndarray, label: str
) -> pd.DataFrame:
    """Posterior mean and 95% band for `lam`, tagged with an experiment label."""
    central = mcmc_samples["lam"].mean(axis=0)
    lower = np.percentile(mcmc_samples["lam"], 2.5, axis=0)
    upper = np.percentile(mcmc_samples["lam"], 97.5, axis=0)
    return pd.DataFrame(
        {
            "x": x,
            "lam": central.flatten(),
            "lower": lower.flatten(),
            "upper": upper.flatten(),
            "experiment": label,
        }
    )


def plot_posterior_band(
    source: pd.DataFrame,
    df_true: pd.DataFrame,
    df_segment: pd.DataFrame,
    title: str | None = None,
) -> alt.Chart:
    """Posterior mean + 95% band, layered over the true curve and raw interval markers."""
    chart = alt.Chart(source).mark_line(color="purple").encode(x="x", y="lam")
    band = (
        alt.Chart(source)
        .mark_area(color="purple", opacity=0.2)
        .encode(x="x", y="lower", y2="upper")
    )
    result = plot_data(df_true, df_segment) + chart + band
    return result.properties(title=title) if title else result

In [31]:
def summarize_posterior_lam_betaIm_median(
    mcmc_samples: dict,
    PrIm: ArrayLike,
    Phi_eff: ArrayLike,
    x: np.ndarray,
    label: str,
) -> pd.DataFrame:
    """
    Posterior median of `lam` computed from `betaIm` only (the identified
    subspace), with the uncertainty band taken from the full posterior `lam`
    -- which comes from jointly fitting `betaIm` and `betaKer` via MCMC, so
    the null-space contribution to the band is data-informed rather than a
    raw prior draw. (A prior-only draw of `betaKer` produced implausibly
    wide bands, suggesting the null space isn't actually as uninformed as
    the Fisher-information analysis assumed -- jointly fitting it lets the
    likelihood constrain it instead.)
    """
    beta0 = mcmc_samples["beta0"]
    betaIm = mcmc_samples["betaIm"]

    lam_Im = jnp.exp(beta0[:, None] + (Phi_eff @ (PrIm @ betaIm.T)).T)
    central = np.median(lam_Im, axis=0)

    lam_full = mcmc_samples["lam"]
    lower = np.percentile(lam_full, 2.5, axis=0)
    upper = np.percentile(lam_full, 97.5, axis=0)

    return pd.DataFrame(
        {
            "x": x,
            "lam": np.asarray(central).flatten(),
            "lower": np.asarray(lower).flatten(),
            "upper": np.asarray(upper).flatten(),
            "experiment": label,
        }
    )

In [48]:
def summarize_posterior_lam_nullspace_prior_uncertainty(
    mcmc_samples: dict,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    Phi_eff: ArrayLike,
    x: np.ndarray,
    label: str,
    rng_key: ArrayLike,
    scale: float = 1.0,
) -> pd.DataFrame:
    """
    Posterior median of `lam` from betaIm only (`mcmc_samples["lam"]`, from a
    model_reduced_bspline run with no betaKer), with an uncertainty band
    built by drawing one fresh betaKer ~ Normal(0, I) per posterior betaIm
    sample and pushing it through Phi_eff -- i.e. null-space uncertainty
    taken from its prior, not fit to data.
    """
    beta0 = mcmc_samples["beta0"]
    betaIm = mcmc_samples["betaIm"]
    S = beta0.shape[0]
    MKer = PrKer.shape[1]

    betaKer_prior = jax.random.normal(rng_key, shape=(S, MKer)) * scale
    lam_full = jnp.exp(
        beta0[:, None] + (Phi_eff @ (PrIm @ betaIm.T + PrKer @ betaKer_prior.T)).T
    )

    central = np.median(mcmc_samples["lam"], axis=0)
    lower = np.percentile(lam_full, 2.5, axis=0)
    upper = np.percentile(lam_full, 97.5, axis=0)

    return pd.DataFrame(
        {
            "x": x,
            "lam": np.asarray(central).flatten(),
            "lower": np.asarray(lower).flatten(),
            "upper": np.asarray(upper).flatten(),
            "experiment": label,
        }
    )

### Experiment 1: Naive B-Spline model

In [33]:
def model_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    K: ArrayLike | None = None,
    y=None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    N: ArrayLike
        Number of observations in each interval.
    interval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    K: ArrayLike | None, optional
        Cholesky factor of the coefficient covariance prior (Sigma = K @ K.T).
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    M = Phi.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    beta = numpyro.sample("w", dist.Normal(0, 1).expand([M]).to_event(1))
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi_eff @ beta))
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [34]:
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

x = np.arange(0, 101)
y = df_model["y"].values
interval_id = df_model["obs_interval_id"].values
N = df_model["N"].values

intervals = df_obs["obs_interval"].unique().tolist()
A = build_interval_sum_matrix(intervals)
interval_width = compute_interval_widths(intervals)

# Convert to JAX arrays
y_jax = jnp.array(y)
interval_id = jnp.array(interval_id)
N = jnp.array(N)
W_jax = jnp.array(W)
Phi_jax = jnp.array(Phi)
K_jax = jnp.array(K)
interval_width_jax = jnp.array(interval_width)

In [35]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_bspline,
    W=W_jax,
    Phi=Phi_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:04<00:00, 221.00it/s, 1023 steps of size 3.78e-03. acc. prob=0.94]

Number of divergences: 0
dict_keys(['beta0', 'lam', 'w'])


In [36]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior_band(source_exp1, df_rates, df_expanded)

alt.LayerChart(...)

### Experiment 2: Reduced subspace model

In [37]:
def model_reduced_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
    K: ArrayLike | None = None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    K: ArrayLike, optional
        Cholesky factor of a coefficient covariance prior (Sigma = K @ K.T),
        e.g. a GMRF smoothness prior. If given, `PrIm`/`PrKer` are assumed to
        live in the whitened coordinates beta = K @ alpha, so the basis is
        effectively Phi @ K instead of Phi. If None (default), no
        reparametrization is applied.
    """
    M = PrIm.shape[1]
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([M]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi_eff @ (PrIm @ betaIm)))
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [38]:
PrIm_jax = jnp.array(PrIm)

In [39]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_reduced_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 1532.72it/s, 127 steps of size 4.69e-02. acc. prob=0.94]

Number of divergences: 0
dict_keys(['beta0', 'betaIm', 'lam'])


In [40]:
source_exp2 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "2. Reduced Subspace"
)
plot_posterior_band(source_exp2, df_rates, df_expanded)

alt.LayerChart(...)

### Experiment 3: Null-space included model

In [41]:
def model_nullinc_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
    K: ArrayLike | None = None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    K: ArrayLike, optional
        Cholesky factor of a coefficient covariance prior (Sigma = K @ K.T),
        e.g. a GMRF smoothness prior. If given, `PrIm`/`PrKer` are assumed to
        live in the whitened coordinates beta = K @ alpha, so the basis is
        effectively Phi @ K instead of Phi. If None (default), no
        reparametrization is applied.
    """
    MIm = PrIm.shape[1]
    MKer = PrKer.shape[1]
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([MIm]).to_event(1))
    betaKer = numpyro.sample("betaKer", dist.Normal(0, 1).expand([MKer]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic(
        "lam", jnp.exp(beta0 + Phi_eff @ (PrIm @ betaIm + PrKer @ betaKer))
    )
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [42]:
PrKer_jax = jnp.array(PrKer)

In [43]:
# MCMC inference with NUTS -- joint inference over betaIm and betaKer, so
# the null-space directions are constrained by the likelihood rather than
# left as pure prior draws. The median of `lam` below still uses betaIm
# only; betaKer's posterior only feeds the uncertainty band.
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_nullinc_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    PrKer=PrKer_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 1073.75it/s, 127 steps of size 4.21e-02. acc. prob=0.95]


Number of divergences: 0
dict_keys(['beta0', 'betaIm', 'betaKer', 'lam'])


In [44]:
source_exp3 = summarize_posterior_lam_betaIm_median(
    mcmc_samples,
    PrIm_jax,
    Phi_jax @ K_jax,
    np.linspace(0, 100, 100),
    "3. Null-Space Included",
)
plot_posterior_band(source_exp3, df_rates, df_expanded)

alt.LayerChart(...)

### Experiment 4: Null-Space Prior-Uncertainty Model

Experiment 2 drops `betaKer` entirely (zero posterior uncertainty in the null space); Experiment 3 jointly fits `betaKer` alongside `betaIm` via MCMC, so its contribution to the uncertainty band is constrained by the likelihood. This experiment instead takes the Fisher-information null space at face value -- genuinely uninformative -- and shows the uncertainty that follows from that assumption alone: infer `betaIm` only (`model_reduced_bspline`, same as Experiment 2), then for each posterior draw of `betaIm` sample a fresh `betaKer` from its `Normal(0,1)` prior and propagate it through `Phi @ K` to build the band. The median `lam` line still comes from `betaIm` alone.

Comparing this against Experiment 3 isolates how much the joint MCMC fit actually narrows the null-space uncertainty relative to just trusting the prior.

In [45]:
# MCMC inference with NUTS -- inference only over betaIm (same model as
# Experiment 2); betaKer's contribution to the uncertainty band is injected
# post-hoc from its prior below, not fit here.
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_reduced_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:00<00:00, 1934.87it/s, 127 steps of size 4.69e-02. acc. prob=0.94]

Number of divergences: 0
dict_keys(['beta0', 'betaIm', 'lam'])


In [46]:
source_exp4 = summarize_posterior_lam_nullspace_prior_uncertainty(
    mcmc_samples,
    PrIm_jax,
    PrKer_jax,
    Phi_jax @ K_jax,
    np.linspace(0, 100, 100),
    "4. Null-Space (Prior Uncertainty)",
    jax.random.PRNGKey(4),
)
plot_posterior_band(source_exp4, df_rates, df_expanded)

alt.LayerChart(...)

## Comparing All Four Experiments

In [49]:
experiment_order = [
    "1. Naive",
    "2. Reduced Subspace",
    "3. Null-Space Included",
    "4. Null-Space (Prior Uncertainty)",
]
sources = {
    "1. Naive": source_exp1,
    "2. Reduced Subspace": source_exp2,
    "3. Null-Space Included": source_exp3,
    "4. Null-Space (Prior Uncertainty)": source_exp4,
}

comparison_charts = [
    plot_posterior_band(sources[name], df_rates, df_expanded, title=name).properties(
        width=200, height=200
    )
    for name in experiment_order
]

alt.hconcat(*comparison_charts).resolve_scale(y="shared")

alt.HConcatChart(...)